In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

np.random.seed(42)

def matern52_2d(X1, X2, sigma_sq, ls):
    """2D Matérn-5/2 kernel with per-dimension length-scales.
    ls = [ℓ_x, ℓ_y] — anisotropic length-scales.
    """
    # Scale each dimension by its length-scale
    X1_scaled = X1 / np.array(ls)
    X2_scaled = X2 / np.array(ls)

    dist = cdist(X1_scaled, X2_scaled, metric='euclidean')
    sqrt5_r = np.sqrt(5) * dist
    return sigma_sq * (1 + sqrt5_r + 5 * dist**2 / 3) * np.exp(-sqrt5_r)

# Generate a grid for visualization
nx, ny = 50, 50
x_grid = np.linspace(0, 4, nx)
y_grid = np.linspace(0, 4, ny)
xx, yy = np.meshgrid(x_grid, y_grid)
X_grid = np.column_stack([xx.ravel(), yy.ravel()])  # Shape: (2500, 2)

sigma_sq = 100.0

# Compare isotropic vs anisotropic
configs = {
    'Isotropic\nℓ_x = ℓ_y = 1.0 km': [1.0, 1.0],
    'Mild anisotropy\nℓ_x=2.0, ℓ_y=1.0': [2.0, 1.0],
    'Strong anisotropy\nℓ_x=3.0, ℓ_y=0.5': [3.0, 0.5],
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, ls) in zip(axes, configs.items()):
    K = matern52_2d(X_grid, X_grid, sigma_sq, ls) + 1e-4 * np.eye(len(X_grid))
    sample = np.random.multivariate_normal(np.zeros(len(X_grid)), K)

    im = ax.contourf(xx, yy, sample.reshape(nx, ny), levels=20, cmap='RdYlGn')
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Easting (km)')
    ax.set_ylabel('Northing (km)')
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax, label='RMR')

plt.suptitle('Isotropic vs Anisotropic GP Prior Draws (2D)', fontsize=13)
plt.tight_layout()
plt.savefig("anisotropy_comparison.png", dpi=150)
plt.show()

print("=== OBSERVATIONS ===")
print("Isotropic:        Circular blobs. Equal correlation in all directions.")
print("Mild anisotropy:  Elliptical blobs, stretched horizontally.")
print("Strong anisotropy: Bands/layers! Looks like geological strata.")
print()
print("→ Strong anisotropy (ℓ_x >> ℓ_y) produces LAYERED spatial fields.")
print("→ This is exactly what real geological cross-sections look like.")
print("→ In PyMC: pm.gp.cov.Matern52(input_dim=2, ls=[ℓ_x, ℓ_y])")

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

np.random.seed(42)

# === MOCK BOREHOLE DATA ===
# 8 boreholes at random map coordinates within a 4km × 4km study area
n_boreholes = 8
x_coords = np.array([0.5, 1.2, 2.0, 0.8, 3.0, 3.5, 1.5, 2.8])  # Easting (km)
y_coords = np.array([0.5, 1.5, 0.8, 2.8, 1.0, 2.5, 3.2, 3.0])  # Northing (km)
rmr_values = np.array([55, 42, 48, 30, 52, 35, 28, 32])           # RMR

# Assemble into 2D input matrix
X_obs = np.column_stack([x_coords, y_coords])  # Shape: (8, 2)
y_obs = rmr_values.astype(float)

print(f"=== 2D BOREHOLE DATA ===")
print(f"Study area: 4 km × 4 km")
print(f"Boreholes:  {n_boreholes}")
print(f"Input shape: {X_obs.shape} (n_samples, 2)")
for i in range(n_boreholes):
    print(f"  BH-{i+1}: ({x_coords[i]:.1f}, {y_coords[i]:.1f}) km  →  RMR = {rmr_values[i]}")

# Zero-center for HSGP (Day 3 lesson!)
X_center = X_obs.mean(axis=0)
X_centered = X_obs - X_center
print(f"\nCenter: ({X_center[0]:.2f}, {X_center[1]:.2f})")
print(f"Centered range: [{X_centered.min():.2f}, {X_centered.max():.2f}]")

# Visualize borehole locations
fig, ax = plt.subplots(figsize=(7, 7))
scatter = ax.scatter(x_coords, y_coords, c=rmr_values, cmap='RdYlGn',
                      s=200, edgecolors='black', linewidth=2, zorder=5,
                      vmin=20, vmax=60)
for i in range(n_boreholes):
    ax.annotate(f'BH-{i+1}\nRMR={rmr_values[i]}',
                (x_coords[i], y_coords[i]),
                textcoords="offset points", xytext=(12, 8),
                fontsize=8, fontweight='bold')
ax.set_xlabel('Easting (km)', fontsize=12)
ax.set_ylabel('Northing (km)', fontsize=12)
ax.set_title('Borehole Locations & RMR Values', fontsize=13)
ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='RMR')
plt.tight_layout()
plt.savefig("borehole_map.png", dpi=150)
plt.show()

In [ ]:
# === 2D HSGP MODEL WITH ANISOTROPIC KERNEL ===
with pm.Model() as spatial_model:

    # LENGTH-SCALES: one per dimension (anisotropic)
    # ℓ_x (Easting): how far does RMR correlate horizontally?
    # ℓ_y (Northing): how far does RMR correlate vertically?
    # We expect ℓ_x ≥ ℓ_y (horizontal correlation ≥ vertical)
    ell_x = pm.InverseGamma("ell_x", alpha=3, beta=3)
    ell_y = pm.InverseGamma("ell_y", alpha=3, beta=3)

    # AMPLITUDE
    eta = pm.HalfCauchy("eta", beta=15)

    # NOISE
    sigma = pm.HalfNormal("sigma", sigma=5)

    # KERNEL: 2D Matérn-5/2 with SEPARATE length-scales
    # input_dim=2: two spatial dimensions
    # ls=[ℓ_x, ℓ_y]: per-dimension length-scales (ARD kernel)
    cov_func = eta**2 * pm.gp.cov.Matern52(input_dim=2, ls=[ell_x, ell_y])

    # HSGP: m=[15, 15] = 225 basis functions total
    # c=1.5 with centered data
    gp = pm.gp.HSGP(
        m=[15, 15],     # 15 basis functions per dimension
        c=1.5,          # Boundary extension
        cov_func=cov_func,
    )

    # Prior on latent GP
    f = gp.prior("f", X=X_centered)

    # Likelihood
    y_ = pm.Normal("y_obs", mu=f, sigma=sigma, observed=y_obs)

print(spatial_model)

# === SAMPLE ===
with spatial_model:
    trace_2d = pm.sample(
        draws=1000, tune=1000, chains=4,
        nuts_sampler="numpyro",
        target_accept=0.9,
        random_seed=42,
    )

# === DIAGNOSTICS ===
print("\n=== POSTERIOR SUMMARY ===")
print(az.summary(trace_2d, var_names=["ell_x", "ell_y", "eta", "sigma"]))

az.plot_trace(trace_2d, var_names=["ell_x", "ell_y", "eta", "sigma"])
plt.suptitle("2D GP Hyperparameter Traces", fontsize=14)
plt.tight_layout()
plt.savefig("2d_gp_traces.png", dpi=150)
plt.show()

# Check anisotropy: is ℓ_x > ℓ_y?
ell_x_post = trace_2d.posterior["ell_x"].values.flatten()
ell_y_post = trace_2d.posterior["ell_y"].values.flatten()
aniso_ratio = ell_x_post / ell_y_post
print(f"\nAnisotropy ratio ℓ_x/ℓ_y: {aniso_ratio.mean():.2f} "
      f"[{np.percentile(aniso_ratio, 5):.2f}, {np.percentile(aniso_ratio, 95):.2f}]")
print("→ Ratio > 1: horizontal correlation is longer (expected for layered geology)")
print("→ Ratio ≈ 1: approximately isotropic (no directional preference)")

In [ ]:
# === PREDICTION GRID ===
# Create a dense 50×50 grid covering the study area
nx, ny = 50, 50
x_grid = np.linspace(0, 4, nx)
y_grid = np.linspace(0, 4, ny)
xx, yy = np.meshgrid(x_grid, y_grid)
X_pred = np.column_stack([xx.ravel(), yy.ravel()])  # Shape: (2500, 2)

# Center the prediction grid (same center as training data!)
X_pred_centered = X_pred - X_center

# === PREDICT ===
with spatial_model:
    f_pred = gp.conditional("f_pred", Xnew=X_pred_centered)
    ppc_2d = pm.sample_posterior_predictive(
        trace_2d, var_names=["f_pred"], random_seed=42
    )

f_2d = ppc_2d.posterior_predictive["f_pred"].values.reshape(-1, nx * ny)
f_mean_2d = f_2d.mean(axis=0).reshape(ny, nx)
f_std_2d = f_2d.std(axis=0).reshape(ny, nx)

# === THE SPATIAL MAP: 4-Panel Figure ===
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# (a) Raw data
ax = axes[0, 0]
scatter = ax.scatter(x_coords, y_coords, c=rmr_values, cmap='RdYlGn',
                      s=200, edgecolors='black', linewidth=2, zorder=5,
                      vmin=20, vmax=60)
ax.set_title('(a) Raw Borehole Data', fontsize=12, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='RMR')

# (b) GP posterior mean
ax = axes[0, 1]
im = ax.contourf(xx, yy, f_mean_2d, levels=20, cmap='RdYlGn', vmin=20, vmax=60)
ax.scatter(x_coords, y_coords, c='black', s=50, marker='^', zorder=5)
ax.set_title('(b) GP Posterior Mean (RMR)', fontsize=12, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.colorbar(im, ax=ax, label='RMR')

# (c) GP posterior std (uncertainty)
ax = axes[1, 0]
im = ax.contourf(xx, yy, f_std_2d, levels=20, cmap='YlOrRd')
ax.scatter(x_coords, y_coords, c='white', s=50, marker='^', zorder=5,
           edgecolors='black')
ax.set_title('(c) Posterior Uncertainty (σ)', fontsize=12, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.colorbar(im, ax=ax, label='Std Dev (RMR units)')

# (d) Coefficient of variation (relative uncertainty)
cv_2d = f_std_2d / np.abs(f_mean_2d) * 100
ax = axes[1, 1]
im = ax.contourf(xx, yy, cv_2d, levels=20, cmap='YlOrRd')
ax.scatter(x_coords, y_coords, c='white', s=50, marker='^', zorder=5,
           edgecolors='black')
ax.set_title('(d) Relative Uncertainty (CV %)', fontsize=12, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.colorbar(im, ax=ax, label='CV (%)')

for ax in axes.flat:
    ax.set_xlabel('Easting (km)')
    ax.set_ylabel('Northing (km)')
    ax.set_xlim(0, 4)
    ax.set_ylim(0, 4)

plt.suptitle('2D Spatial GP: Borehole RMR → Continuous Map with Uncertainty', fontsize=14)
plt.tight_layout()
plt.savefig("spatial_rmr_map.png", dpi=150)
plt.show()

print("=== WHAT YOU JUST PRODUCED ===")
print("(a) 8 scattered borehole points — all you started with.")
print("(b) A CONTINUOUS RMR field across the entire 4×4 km study area.")
print("(c) An UNCERTAINTY map — bright (high σ) where no boreholes, dark near data.")
print("(d) RELATIVE uncertainty — where is the GP most ignorant relative to the value?")
print()
print("→ Panel (c) IS the answer to 'where should we drill next?'")
print("→ Panel (b) + (c) together are what you put in a paper or report.")
print("→ No other method gives you BOTH the prediction AND the honest uncertainty.")


In [ ]:
# === FORCE ISOTROPY: Set ℓ_x = ℓ_y ===
# Compare against the anisotropic model to see if data supports anisotropy.

with pm.Model() as iso_model:
    ell = pm.InverseGamma("ell", alpha=3, beta=3)  # SHARED length-scale
    eta = pm.HalfCauchy("eta", beta=15)
    sigma = pm.HalfNormal("sigma", sigma=5)

    # ISOTROPIC: same ℓ in both dimensions
    cov_func = eta**2 * pm.gp.cov.Matern52(input_dim=2, ls=ell)

    gp_iso = pm.gp.HSGP(m=[15, 15], c=1.5, cov_func=cov_func)
    f = gp_iso.prior("f", X=X_centered)
    y_ = pm.Normal("y_obs", mu=f, sigma=sigma, observed=y_obs)

    trace_iso = pm.sample(draws=1000, tune=1000, chains=2,
                          nuts_sampler="numpyro", target_accept=0.9,
                          random_seed=42)

    f_pred_iso = gp_iso.conditional("f_pred", Xnew=X_pred_centered)
    ppc_iso = pm.sample_posterior_predictive(
        trace_iso, var_names=["f_pred"], random_seed=42
    )

f_iso = ppc_iso.posterior_predictive["f_pred"].values.reshape(-1, nx * ny)
f_iso_mean = f_iso.mean(axis=0).reshape(ny, nx)
f_iso_std = f_iso.std(axis=0).reshape(ny, nx)

# Compare
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].contourf(xx, yy, f_iso_mean, levels=20, cmap='RdYlGn', vmin=20, vmax=60)
axes[0, 0].scatter(x_coords, y_coords, c='black', s=50, marker='^', zorder=5)
axes[0, 0].set_title('Isotropic: Mean', fontweight='bold')

axes[0, 1].contourf(xx, yy, f_mean_2d, levels=20, cmap='RdYlGn', vmin=20, vmax=60)
axes[0, 1].scatter(x_coords, y_coords, c='black', s=50, marker='^', zorder=5)
axes[0, 1].set_title('Anisotropic: Mean', fontweight='bold')

axes[1, 0].contourf(xx, yy, f_iso_std, levels=20, cmap='YlOrRd')
axes[1, 0].scatter(x_coords, y_coords, c='white', s=50, marker='^', zorder=5,
                    edgecolors='black')
axes[1, 0].set_title('Isotropic: Uncertainty', fontweight='bold')

axes[1, 1].contourf(xx, yy, f_std_2d, levels=20, cmap='YlOrRd')
axes[1, 1].scatter(x_coords, y_coords, c='white', s=50, marker='^', zorder=5,
                    edgecolors='black')
axes[1, 1].set_title('Anisotropic: Uncertainty', fontweight='bold')

for ax in axes.flat:
    ax.set_aspect('equal')
    ax.set_xlim(0, 4)
    ax.set_ylim(0, 4)
    ax.grid(True, alpha=0.3)

plt.suptitle('Isotropic vs Anisotropic GP: Same Data, Different Assumptions', fontsize=13)
plt.tight_layout()
plt.savefig("iso_vs_aniso.png", dpi=150)
plt.show()

# Model comparison via LOO
print("\n=== MODEL COMPARISON (LOO-CV) ===")
# If your data supports anisotropy, the anisotropic model will have
# higher ELPD (expected log predictive density).
# With only 8 boreholes, the difference may be small — the data
# may not have enough power to distinguish ℓ_x from ℓ_y.
print("With 8 boreholes, anisotropy may not be detectable.")
print("With 20+ boreholes across varied terrain, it usually is.")
print("RECOMMENDATION: Always START with anisotropic. If the posterior")
print("of ℓ_x and ℓ_y overlap heavily, isotropy is sufficient.")

In [ ]:
# === EXTRACTING A 1D PROFILE FROM THE 2D MAP ===
# Your 2D map covers the study area. But your tunnel follows a specific path.
# Extract the GP prediction ALONG the tunnel alignment.

# Define a tunnel alignment: straight line from (0.5, 1.0) to (3.5, 3.0)
n_tunnel = 100
t = np.linspace(0, 1, n_tunnel)
tunnel_x = 0.5 + 3.0 * t   # Easting
tunnel_y = 1.0 + 2.0 * t   # Northing
chainage = np.sqrt((tunnel_x - tunnel_x[0])**2 + (tunnel_y - tunnel_y[0])**2)

X_tunnel = np.column_stack([tunnel_x, tunnel_y])
X_tunnel_centered = X_tunnel - X_center

# Get predictions along tunnel
with spatial_model:
    f_tunnel = gp.conditional("f_tunnel", Xnew=X_tunnel_centered)
    ppc_tunnel = pm.sample_posterior_predictive(
        trace_2d, var_names=["f_tunnel"], random_seed=42
    )

f_t = ppc_tunnel.posterior_predictive["f_tunnel"].values.reshape(-1, n_tunnel)
f_t_mean = f_t.mean(axis=0)
f_t_std = f_t.std(axis=0)
f_t_q025 = np.percentile(f_t, 2.5, axis=0)
f_t_q975 = np.percentile(f_t, 97.5, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Map with tunnel alignment
ax = axes[0]
ax.contourf(xx, yy, f_mean_2d, levels=20, cmap='RdYlGn', alpha=0.7, vmin=20, vmax=60)
ax.plot(tunnel_x, tunnel_y, 'k-', linewidth=3, label='Tunnel alignment')
ax.scatter(x_coords, y_coords, c=rmr_values, cmap='RdYlGn', s=150,
           edgecolors='black', linewidth=2, zorder=5, vmin=20, vmax=60)
ax.set_title('2D Map with Tunnel Alignment', fontsize=12, fontweight='bold')
ax.set_aspect('equal')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: 1D profile along tunnel
ax = axes[1]
ax.fill_between(chainage, f_t_q025, f_t_q975, alpha=0.3, color='steelblue',
                label='95% CI')
ax.plot(chainage, f_t_mean, 'b-', linewidth=2, label='GP mean')

# Mark where boreholes are near the tunnel
for i in range(n_boreholes):
    dist_to_tunnel = np.min(np.sqrt((tunnel_x - x_coords[i])**2 +
                                     (tunnel_y - y_coords[i])**2))
    if dist_to_tunnel < 0.5:  # Within 500m of tunnel
        ch_idx = np.argmin(np.sqrt((tunnel_x - x_coords[i])**2 +
                                    (tunnel_y - y_coords[i])**2))
        ax.scatter(chainage[ch_idx], rmr_values[i], c='red', s=100, zorder=5,
                   edgecolors='black', label=f'BH-{i+1} (nearby)' if i == 0 else None)

ax.set_xlabel('Chainage (km)', fontsize=12)
ax.set_ylabel('RMR', fontsize=12)
ax.set_title('RMR Profile Along Tunnel Alignment', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('2D Map → 1D Tunnel Profile (Cross-Section Extraction)', fontsize=13)
plt.tight_layout()
plt.savefig("tunnel_cross_section.png", dpi=150)
plt.show()

print("=== THE POWER OF 2D GP ===")
print("You fit ONE 2D GP to ALL boreholes in the study area.")
print("Then you EXTRACT a 1D profile along ANY path — tunnel, road, pipeline.")
print("The uncertainty reflects how far the path is from the nearest borehole.")
print("Sections of tunnel far from any borehole have WIDE bands.")